# AirSense V1 — Exploratory Data Analysis

**Phase 3 of the V1 research protocol.** Beijing PM2.5, 2010–2014.

> This notebook is a **presentation layer**. Every number and figure shown
> here is produced by `src/analysis/eda.py`, the same module that
> `scripts/run_eda.py` runs. The artifacts written to `results/eda/`,
> `figures/` and `artifacts/eda_summary.json` are **authoritative**; this
> notebook reads them rather than recomputing them independently, so the
> two can never drift apart.

**Runtime:** CPython 3.6.7 with pandas 0.23.4, numpy 1.15.4, scipy 1.1.0,
matplotlib 3.0.2, seaborn 0.9.0 — the frozen pre-2019-04-26 stack.

**Out of scope for this phase:** no cleaning, no imputation, no encoding,
no feature engineering, no train/test split, and no model. Those belong to
later phases.

## 1. Objective

> *What does the historical Beijing PM2.5 dataset contain, what are its
> important statistical properties, how does PM2.5 vary temporally and with
> meteorological conditions, and what data-quality issues must be resolved
> before predictive modelling?*

The raw CSV is opened read-only. Rows with a missing target are filtered
**in memory** for individual statistics; no cleaned dataset is written.

In [ ]:
import os, sys, json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analysis import eda
from src.data.audit import sha256_of_file

RESULTS = os.path.join(PROJECT_ROOT, "results", "eda")
FIGURES = os.path.join(PROJECT_ROOT, "figures")

def table(name):
    """Read one authoritative result table written by scripts/run_eda.py."""
    return pd.read_csv(os.path.join(RESULTS, name))

def figure(name):
    display(Image(filename=os.path.join(FIGURES, name)))

with open(os.path.join(PROJECT_ROOT, "artifacts", "eda_summary.json")) as fh:
    summary = json.load(fh)

print("python  :", "%d.%d.%d" % sys.version_info[:3])
print("pandas  :", pd.__version__)
print("dataset :", summary["dataset"]["sha256"])

### Input integrity gate

The analysis refuses to run against an unexpected dataset. The digest below
must match the value frozen in `docs/DATASET_AUDIT.md`.

In [ ]:
raw = os.path.join(PROJECT_ROOT, "data", "raw", eda.RAW_FILENAME)
observed = sha256_of_file(raw)
print("expected:", eda.EXPECTED_SHA256)
print("observed:", observed)
print("match   :", observed == eda.EXPECTED_SHA256)

## 2. Dataset overview

43,824 hourly observations, 13 columns, 2010-01-01 00:00 to 2014-12-31 23:00.
PM2.5 was measured at the US Embassy in Beijing; the meteorological
variables come from Beijing Capital International Airport.

In [ ]:
print("rows    :", summary["dataset"]["row_count"])
print("columns :", summary["dataset"]["column_count"])
print("period  :", summary["dataset"]["first_timestamp"], "->",
      summary["dataset"]["last_timestamp"])
print("years   :", summary["dataset"]["n_years"])
print("")
print("observations per year:")
for year, count in sorted(summary["dataset"]["observations_per_year"].items()):
    print("   %s: %d" % (year, count))

table("dataset_profile.csv")

## 3. Data quality

Two questions matter here: what is missing, and is the hourly timeline
actually continuous? The second question is decisive for the chronological
evaluation the protocol requires.

In [ ]:
display(table("missing_values.csv"))
display(table("time_series_integrity.csv"))

### Missingness pattern

Missing values occur **only** in the target. The important question is
whether they are spread evenly through time — if they are not, an early
training period and a later test period would carry different amounts of
missing data.

In [ ]:
display(table("pm25_missingness_by_year.csv"))
display(table("pm25_missingness_by_month.csv"))
figure("eda_pm25_missingness.png")

## 4. PM2.5 distribution

The right-hand panel below uses a log10 **display** scale. The target
variable itself is not transformed — no modelling transformation has been
approved.

In [ ]:
display(table("pm25_descriptive_statistics.csv"))
figure("eda_pm25_distribution.png")

## 5. Temporal analysis

PM2.5 grouped by year, calendar month and hour of day. Counts are given
alongside means so that group sizes are visible.

In [ ]:
display(table("pm25_by_year.csv"))
figure("eda_pm25_yearly.png")

In [ ]:
display(table("pm25_by_month.csv"))
figure("eda_pm25_monthly.png")

In [ ]:
display(table("pm25_by_hour.csv"))
figure("eda_pm25_hourly.png")

### Seasonal grouping

Meteorological seasons, mapped from calendar month:

| Season | Months |
|---|---|
| Winter | December, January, February |
| Spring | March, April, May |
| Summer | June, July, August |
| Autumn | September, October, November |

Mean and median are read together deliberately: they do not tell the same
story here.

In [ ]:
display(table("pm25_by_season.csv"))
figure("eda_pm25_seasonal.png")

## 6. Meteorological relationships

Shown as binned summaries rather than a 40k-point scatter, which would be
an unreadable block of ink. Every available observation contributes to the
bins — nothing is sampled, so these are genuine summaries and the figure is
deterministic.

In [ ]:
display(table("numerical_feature_summary.csv"))
figure("eda_pm25_meteorology.png")

## 7. Wind direction (categorical)

`cbwd` is left exactly as distributed. No ordinal or one-hot encoding is
applied — encoding is a preprocessing decision, not an EDA one.

In [ ]:
display(table("pm25_by_wind_direction.csv"))
figure("eda_pm25_wind_direction.png")

## 8. Correlation analysis

Pearson over pairwise-complete observations, plus Spearman as a robustness
check against the skew of PM2.5. Correlation is **not** causation, and no
feature is selected or removed on the basis of these values.

In [ ]:
display(table("pearson_correlation.csv"))
display(table("pm25_spearman_correlations.csv"))
figure("eda_correlation_heatmap.png")

## 9. Observations

The written analysis lives in [`docs/EDA_ANALYSIS.md`](../docs/EDA_ANALYSIS.md),
which states only what the tables above actually support. In brief:

- PM2.5 is strongly right-skewed, with a mean well above its median.
- The hourly timeline is **complete** — no gap, no duplicate.
- Missing targets are **not** evenly distributed across years.
- Cumulative wind speed has the strongest monotonic association with PM2.5
  of any meteorological variable measured here, and it is negative.
- The meteorological predictors are strongly correlated with one another.

No causal claim is made, and no model has been fitted.

## 10. Limitations and next step

**Limitations.** A single monitoring site in one city; a fixed 2010–2014
window; hourly resolution only; emissions, traffic and industrial activity
are unmeasured, so meteorology alone cannot explain concentration; and
`Iws`/`Is`/`Ir` are cumulative within a spell rather than instantaneous
readings.

**Next step — Phase 4, data cleaning and preprocessing.** The decisions this
phase hands forward are recorded in the *Implications for Preprocessing*
section of `docs/EDA_ANALYSIS.md`. Nothing has been cleaned, imputed, encoded
or split yet.